In [ ]:
!pip install -q datasets sentence-transformers scikit-learn xgboost torch scipy kaggle fastapi uvicorn httpx
import os, json, re, torch
import numpy as np
import pandas as pd
os.makedirs("artifacts", exist_ok=True)
print("CUDA GPU available:", torch.cuda.is_available(), "| Device:", torch.cuda.get_device_name(0) if
torch.cuda.is_available() else "CPU")

CUDA GPU available: True | Device: Tesla T4


In [ ]:
from datasets import load_dataset
import re
import numpy as np
import pandas as pd

# Stream Parquet archive without crashing Colab RAM
ds = load_dataset("Bob-68/cfpb-consumer-complaints", split="train", streaming=True)
first_keys = list(next(iter(ds)).keys())
print("Detected CFPB columns:", first_keys)

TARGET_PRODUCTS = {
    "Checking or savings account",
    "Credit card",
    "Credit card or prepaid card",
    "Money transfer, virtual currency, or money service",
    "Debt collection",
}
VALID_RESP = {
    "Closed with monetary relief",
    "Closed with explanation",
    "Closed with non-monetary relief",
    "Closed",
}

rows = []
for i, r in enumerate(ds):
    resp = r.get("company_response_to_consumer") or r.get("company_response")
    narr = r.get("consumer_complaint_narrative")
    dt = str(r.get("date_received", ""))[:10]
    if narr and r.get("product") in TARGET_PRODUCTS and dt >= "2022-01-01" and resp in VALID_RESP:
        rows.append({
            "date_received": dt,
            "product": r.get("product"),
            "issue": str(r.get("issue") or "Other"),
            "company": str(r.get("company") or "Unknown"),
            "state": str(r.get("state") or "NA"),
            "tags": str(r.get("tags") or ""),
            "narrative": narr,
            "y": 1 if resp == "Closed with monetary relief" else 0,
        })
    if len(rows) >= 45000 or i >= 850000:
        break

df = pd.DataFrame(rows)
df["date_received"] = pd.to_datetime(df["date_received"])
df = df.sort_values("date_received").reset_index(drop=True)

# Held-out Protected / Proxy Attributes (NEVER passed into model inference inputs!)
df["is_older"] = df["tags"].str.contains("Older American", case=False, na=False).astype(int)
LOW_INC_STATES = {"MS", "WV", "AR", "LA", "NM", "KY", "AL", "OK", "SC", "TN"}
df["is_low_inc_state"] = df["state"].isin(LOW_INC_STATES).astype(int)
if df["is_older"].sum() >= 800:
    df["protected_group"] = df["is_older"]
    PROXY_NAME = "Older American Tag (withheld from inference)"
else:
    df["protected_group"] = ((df["is_older"] == 1) | (df["is_low_inc_state"] == 1)).astype(int)
    PROXY_NAME = "Older American / Lower-Income State Proxy (withheld from inference)"

def pii_parity_scrub(text: str) -> str:
    text = re.sub(r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b", "XX/XX/XXXX", text)
    text = re.sub(r"(?<!\{)\$\s?(\d+(?:,\d{3})*(?:\.\d{2})?)(?!\})", r"{\$\1}", text)
    text = re.sub(r"\b\d{4}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b", "XXXX-XXXX-XXXX-XXXX", text)
    text = re.sub(r"\b\d{3}-\d{2}-\d{4}\b", "XXX-XX-XXXX", text)
    text = re.sub(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b", "XXXX@XXXX.XXX", text)
    return text.strip()

df["narrative_clean"] = df["narrative"].astype(str).apply(pii_parity_scrub)

def extract_evidence_signals(text: str):
    low = text.lower()
    return [
        1.0 if ("{$" in text or "$" in text) else 0.0,
        1.0 if ("xx/xx" in low or re.search(r"\b20\d{2}\b", low)) else 0.0,
        1.0 if any(k in low for k in ["reference", "case #", "ticket", "claim", "dispute number", "confirmation", "#xxxx"]) else 0.0,
        1.0 if any(k in low for k in ["statement", "receipt", "screenshot", "attached", "contract", "email", "letter", "invoice"]) else 0.0,
        1.0 if any(k in low for k in ["unauthorized", "fraud", "regulation e", "reg e", "billing error", "frozen", "duplicate", "fcba"]) else 0.0,
        min(len(text.split()) / 300.0, 1.0),
    ]

ev_mat = np.array([extract_evidence_signals(t) for t in df["narrative_clean"]], dtype=np.float32)

# Strict Chronological Split (65% Train | 15% Conformal Calibration | 20% Future Unseen Test)
n = len(df)
i_tr, i_cal = int(n * 0.65), int(n * 0.80)
prior_mean = float(df["y"].iloc[:i_tr].mean())
alpha_prior = 10.0  # Empirical-Bayes shrinkage weight

comp_hist = {}
causal_feats = []
for _, row in df.iterrows():
    key = (row["company"], row["product"])
    dt = row["date_received"]
    st = comp_hist.get(key, {"cnt": 0, "pos": 0.0, "ef": prior_mean, "es": prior_mean, "dates": []})

    # READ strictly BEFORE observing current row's outcome y
    eb_rate = (st["pos"] + alpha_prior * prior_mean) / (st["cnt"] + alpha_prior)
    d30 = sum(1 for d in st["dates"] if (dt - d).days <= 30)
    d180 = sum(1 for d in st["dates"] if (dt - d).days <= 180)
    surge = (d30 + 1.0) / ((d180 / 6.0) + 1.0)
    causal_feats.append([eb_rate, st["ef"], st["es"], np.log1p(st["cnt"]), surge])

    # UPDATE state AFTER reading
    yv = float(row["y"])
    st["cnt"] += 1
    st["pos"] += yv
    st["ef"] = 0.85 * st["ef"] + 0.15 * yv
    st["es"] = 0.96 * st["es"] + 0.04 * yv
    st["dates"].append(dt)
    if len(st["dates"]) > 400:
        st["dates"] = [d for d in st["dates"] if (dt - d).days <= 180]
    comp_hist[key] = st

tab_mat = np.hstack([np.array(causal_feats, dtype=np.float32), ev_mat])
y_tr, y_cal, y_te = df["y"].values[:i_tr], df["y"].values[i_tr:i_cal], df["y"].values[i_cal:]
X_tab_tr, X_tab_cal, X_tab_te = tab_mat[:i_tr], tab_mat[i_tr:i_cal], tab_mat[i_cal:]

print(f"Loaded {df.shape[0]} rows | Dates: {df['date_received'].min().date()} -> {df['date_received'].max().date()}")
print(f"Relief Rate: {df['y'].mean():.4f} | Proxy Group ({PROXY_NAME}): {int(df['protected_group'].sum())} rows")


README.md:   0%|          | 0.00/1.73k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

Detected CFPB columns: ['date_received', 'product', 'sub_product', 'issue', 'sub_issue', 'consumer_complaint_narrative', 'company_public_response', 'company', 'state', 'zip_code', 'tags', 'submitted_via', 'date_sent_to_company', 'company_response_to_consumer', 'timely_response', 'complaint_id']
Loaded 9659 rows | Dates: 2023-12-05 -> 2026-06-17
Relief Rate: 0.0904 | Proxy Group (Older American / Lower-Income State Proxy (withheld from inference)): 1541 rows


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import average_precision_score, precision_recall_fscore_support, brier_score_loss, accuracy_score
from scipy.sparse import hstack
import xgboost as xgb

gate2_benchmarks = []

def log_eval(name, yt, probs, thr=0.25):
    preds = (probs >= thr).astype(int)
    pr = float(average_precision_score(yt, probs))
    p, r, f1, _ = precision_recall_fscore_support(yt, preds, average="binary", zero_division=0)
    br = float(brier_score_loss(yt, np.clip(probs, 1e-6, 1 - 1e-6)))
    acc = float(accuracy_score(yt, preds))
    print(f"{name:36s} | PR-AUC={pr:.4f} | P={p:.3f} R={r:.3f} F1={f1:.3f} | Brier={br:.4f} | Acc={acc:.3f}")
    gate2_benchmarks.append({
        "model": name,
        "pr_auc": round(pr, 4),
        "precision": round(float(p), 3),
        "recall": round(float(r), 3),
        "f1": round(float(f1), 3),
        "brier": round(br, 4),
        "acc": round(acc, 3)
    })

# B1: Majority Rule (always predict "No Monetary Relief")
log_eval("B1: Majority Rule (All 0)", y_te, np.zeros_like(y_te, dtype=float))

# B2: Simple Financial Heuristic (Company trailing Empirical-Bayes relief rate)
log_eval("B2: Company Trailing Rate", y_te, X_tab_te[:, 0], thr=0.20)

# B3: Published Literature Paradigm (TF-IDF + Causal Tabular + XGBoost)
vec = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=5)
T_tr = vec.fit_transform(df["narrative_clean"].iloc[:i_tr])
T_te = vec.transform(df["narrative_clean"].iloc[i_cal:])
spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)

xgb_b3 = xgb.XGBClassifier(
    n_estimators=250,
    max_depth=5,
    learning_rate=0.06,
    scale_pos_weight=spw,
    tree_method="hist",
    random_state=42
)
xgb_b3.fit(hstack([T_tr, X_tab_tr]), y_tr)
log_eval("B3: TF-IDF + Tab XGBoost", y_te, xgb_b3.predict_proba(hstack([T_te, X_tab_te]))[:, 1], thr=0.50)

B1: Majority Rule (All 0)            | PR-AUC=0.1004 | P=0.000 R=0.000 F1=0.000 | Brier=0.1004 | Acc=0.900
B2: Company Trailing Rate            | PR-AUC=0.2895 | P=0.277 R=0.588 F1=0.376 | Brier=0.0781 | Acc=0.804
B3: TF-IDF + Tab XGBoost             | PR-AUC=0.3198 | P=0.301 R=0.469 F1=0.367 | Brier=0.1014 | Acc=0.837


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
from torch.utils.data import TensorDataset, DataLoader
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)

Z_text = embedder.encode(
    df["narrative_clean"].tolist(),
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
).astype(np.float32)

mu2, sd2 = X_tab_tr.mean(0), X_tab_tr.std(0) + 1e-6
X_tab_n = ((tab_mat - mu2) / sd2).astype(np.float32)

class GRLFn(Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lam * grad_output, None

def dcor_loss(h, s):
    s = s.view(-1, 1).float()
    a = torch.sqrt(torch.cdist(h, h, p=2).pow(2) + 1e-8)
    b = torch.sqrt(torch.cdist(s, s, p=2).pow(2) + 1e-8)
    A = a - a.mean(1, keepdim=True) - a.mean(0, keepdim=True) + a.mean()
    B = b - b.mean(1, keepdim=True) - b.mean(0, keepdim=True) + b.mean()
    return torch.clamp((A * B).mean(), min=0.0) / torch.sqrt((A * A).mean() * (B * B).mean() + 1e-8)

class VeriRailGate2(nn.Module):
    def __init__(self, text_dim=384, tab_dim=11, hid=128):
        super().__init__()
        self.text_proj = nn.Sequential(nn.Linear(text_dim, hid), nn.LayerNorm(hid), nn.GELU())
        self.tab_proj = nn.Sequential(nn.Linear(tab_dim, hid), nn.LayerNorm(hid), nn.GELU())
        self.film_gamma = nn.Linear(hid, hid)
        self.film_beta = nn.Linear(hid, hid)
        self.head = nn.Sequential(nn.Linear(hid, 64), nn.GELU(), nn.Dropout(0.2), nn.Linear(64, 1))
        self.adv = nn.Sequential(
            nn.utils.parametrizations.spectral_norm(nn.Linear(hid, 64)),
            nn.LeakyReLU(0.2),
            nn.utils.parametrizations.spectral_norm(nn.Linear(64, 1))
        )

    def encode(self, z_txt, x_tab):
        ht, hb = self.text_proj(z_txt), self.tab_proj(x_tab)
        gamma = torch.sigmoid(self.film_gamma(hb)) * 2.0
        beta = self.film_beta(hb)
        return F.layer_norm(gamma * ht + beta + hb, (ht.size(-1),))

    def forward(self, z_txt, x_tab, grl_lam=1.0):
        h = self.encode(z_txt, x_tab)
        return self.head(h).squeeze(-1), self.adv(GRLFn.apply(h, grl_lam)).squeeze(-1), h

def train_g2(use_debias=True, epochs=12):
    torch.manual_seed(42)
    m = VeriRailGate2(Z_text.shape[1], X_tab_n.shape[1]).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4)
    dl = DataLoader(
        TensorDataset(
            torch.tensor(Z_text[:i_tr]),
            torch.tensor(X_tab_n[:i_tr]),
            torch.tensor(y_tr, dtype=torch.float32),
            torch.tensor(df["protected_group"].values[:i_tr], dtype=torch.float32)
        ),
        batch_size=256,
        shuffle=True
    )
    pw = min((y_tr == 0).sum() / max((y_tr == 1).sum(), 1), 6.0)
    for epoch in range(epochs):
        m.train()
        for bz, bt, by, bs in dl:
            bz, bt, by, bs = bz.to(device), bt.to(device), by.to(device), bs.to(device)
            lg, al, h = m(bz, bt, grl_lam=(1.0 if use_debias else 0.0))
            bce = F.binary_cross_entropy_with_logits(lg, by, reduction="none")
            lm = (torch.where(by == 1, pw, 1.0) * ((1 - torch.exp(-bce)) ** 1.5) * bce).mean()
            loss = lm + (0.5 * F.binary_cross_entropy_with_logits(al, bs) + 0.35 * dcor_loss(h, bs) if use_debias else 0.0)
            opt.zero_grad(); loss.backward(); opt.step()

    m.eval()
    with torch.no_grad():
        pc = torch.sigmoid(m(torch.tensor(Z_text[i_tr:i_cal]).to(device), torch.tensor(X_tab_n[i_tr:i_cal]).to(device))[0]).cpu().numpy()
        pt = torch.sigmoid(m(torch.tensor(Z_text[i_cal:]).to(device), torch.tensor(X_tab_n[i_cal:]).to(device))[0]).cpu().numpy()
    return m, pc, pt

_, _, p_abl_te = train_g2(use_debias=False, epochs=12)
log_eval("Ablation: FiLM Only (No GRL/dCor)", y_te, p_abl_te, thr=0.45)
m_full, p_cal2, p_te2 = train_g2(use_debias=True, epochs=12)
log_eval("Full VeriRail Gate 2 (FiLM+GRL+dCor)", y_te, p_te2, thr=0.45)

def fit_mondrian(probs, y_true, groups, alpha=0.10, decay=0.004):
    w_all = np.exp(-decay * np.arange(len(y_true))[::-1])
    q_out = {}
    for g in [0, 1]:
        mask = (groups == g)
        if mask.sum() < 30: mask = np.ones(len(y_true), dtype=bool)
        pg, yg, wg = probs[mask], y_true[mask], w_all[mask]
        scores = np.where(yg == 1, 1.0 - pg, pg)
        oi = np.argsort(scores)
        wn = wg[oi] / (wg[oi].sum() + 1.0)
        idx = min(np.searchsorted(np.cumsum(wn), 1.0 - alpha, side="left"), len(oi) - 1)
        q_out[str(g)] = float(scores[oi][idx])
    return q_out

g_cal2, g_te2 = df["protected_group"].values[i_tr:i_cal], df["protected_group"].values[i_cal:]
q2 = fit_mondrian(p_cal2, y_cal, g_cal2, alpha=0.10)
cov0 = float(np.mean([(y_te[i]==1 and (1-p_te2[i])<=q2["0"]) or (y_te[i]==0 and p_te2[i]<=q2["0"]) for i in np.where(g_te2==0)[0]]))
cov1 = float(np.mean([(y_te[i]==1 and (1-p_te2[i])<=q2["1"]) or (y_te[i]==0 and p_te2[i]<=q2["1"]) for i in np.where(g_te2==1)[0]]))
print(f"Mondrian 90% Coverage -> G0={cov0:.3f} | G1={cov1:.3f} | q={q2}")

# Matched-Narrative Counterfactual Proxy Audit
df_te = df.iloc[i_cal:].copy().reset_index(drop=True)
df_te["p_abl"], df_te["p_full"] = p_abl_te, p_te2
Z_te = Z_text[i_cal:]
rgaps, agaps, fgaps = [], [], []
for ip in np.where(df_te["protected_group"].values == 1)[0][:500]:
    rp = df_te.iloc[ip]
    cands = [j for j in np.where(df_te["protected_group"].values == 0)[0] if df_te.iloc[j]["company"]==rp["company"] and df_te.iloc[j]["product"]==rp["product"]]
    if not cands: continue
    sims = Z_te[cands] @ Z_te[ip]; bp = int(np.argmax(sims))
    if sims[bp] >= 0.85:
        j = cands[bp]
        rgaps.append(rp["y"] - df_te.iloc[j]["y"])
        agaps.append(rp["p_abl"] - df_te.iloc[j]["p_abl"])
        fgaps.append(rp["p_full"] - df_te.iloc[j]["p_full"])

failures = []
for k, idx in enumerate(list(df_te[df_te["y"]==0].sort_values("p_full", ascending=False).index[:2]) + list(df_te[df_te["y"]==1].sort_values("p_full", ascending=True).index[:2])):
    r = df_te.iloc[idx]
    failures.append({
        "case_type": "False Positive" if k < 2 else "False Negative",
        "company": r["company"], "product": r["product"],
        "true_y": int(r["y"]), "pred_p": round(float(r["p_full"]), 3),
        "text_snippet": r["narrative_clean"][:240] + "..."
    })

torch.save(m_full.state_dict(), "artifacts/gate2_verirail.pt")
top_cos = df["company"].value_counts().head(40).index.tolist()
comp_lookup = {}
for key, st in comp_hist.items():
    if key[0] in top_cos:
        eb = (st["pos"] + alpha_prior * prior_mean) / (st["cnt"] + alpha_prior)
        d30 = len(st["dates"])
        surge = (min(d30, 50) + 1.0) / ((max(d30, 1) / 6.0) + 1.0)
        comp_lookup[f"{key[0]}||{key[1]}"] = [float(eb), float(st["ef"]), float(st["es"]), float(np.log1p(st["cnt"])), float(surge)]

with open("artifacts/gate2_artifacts.json", "w") as f:
    json.dump({
        "mu": mu2.tolist(), "sd": sd2.tolist(), "q_thresholds": q2,
        "top_companies": top_cos, "products": sorted(list(TARGET_PRODUCTS)),
        "company_stats": comp_lookup, "default_stats": [prior_mean, prior_mean, prior_mean, 0.0, 1.0],
        "proxy_axis": PROXY_NAME, "prior_mean": prior_mean
    }, f, indent=2)
print("STAGE 3 COMPLETE! Gate 2 neural model trained and artifacts exported.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/76 [00:00<?, ?it/s]

Ablation: FiLM Only (No GRL/dCor)    | PR-AUC=0.3476 | P=0.328 R=0.500 F1=0.396 | Brier=0.0936 | Acc=0.847
Full VeriRail Gate 2 (FiLM+GRL+dCor) | PR-AUC=0.3504 | P=0.319 R=0.634 F1=0.424 | Brier=0.0982 | Acc=0.827
Mondrian 90% Coverage -> G0=0.886 | G1=0.924 | q={'0': 0.5614466071128845, '1': 0.6775871515274048}
STAGE 3 COMPLETE! Gate 2 neural model trained and artifacts exported.


In [10]:
# === COLAB CELL 5 (Gate 1: IEEE-CIS + Closed-Loop Coupling + Export) ===
!pip install -U -q kaggle
import os
os.environ["KAGGLE_API_TOKEN"] = "PASTE_TOKEN_HERE"
!mkdir -p ~/.kaggle && echo -n "PASTE_TOKEN_HERE" > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token
!kaggle competitions download -c ieee-fraud-detection -f train_transaction.csv -p /tmp
!if [ -f /tmp/train_transaction.csv.zip ]; then unzip -oq /tmp/train_transaction.csv.zip -d /tmp; fi
!ls -lh /tmp/train_transaction.csv

use_cols = ["TransactionDT", "isFraud", "TransactionAmt", "ProductCD", "card1", "addr1", "addr2", "dist1", "P_emaildomain", "C1", "D1"]
df1 = pd.read_csv("/tmp/train_transaction.csv", usecols=use_cols, nrows=120000).sort_values("TransactionDT").reset_index(drop=True)
df1["C1"] = df1["C1"].fillna(0.0)
df1["D1"] = df1["D1"].fillna(0.0)
df1["dist1"] = df1["dist1"].fillna(0.0)

FREE_EMAILS = {"gmail.com", "yahoo.com", "hotmail.com", "outlook.com", "protonmail.com", "aol.com"}
df1["is_digital_nonlocal"] = (
    df1["P_emaildomain"].isin(FREE_EMAILS) & ((df1["dist1"] > 50) | (df1["ProductCD"].isin(["C", "H", "S"])))
).astype(int)

n1 = len(df1)
i1_tr, i1_cal = int(n1 * 0.65), int(n1 * 0.80)
g1_prior = float(df1["isFraud"].iloc[:i1_tr].mean())
card_hist, feats1 = {}, []
prod_map = {"W": 0, "H": 1, "C": 2, "S": 3, "R": 4}

for _, row in df1.iterrows():
    c = int(row["card1"])
    st = card_hist.get(c, {"cnt": 0, "pos": 0.0, "amt_sum": 0.0})
    eb_f = (st["pos"] + 10.0 * g1_prior) / (st["cnt"] + 10.0)
    avg_a = st["amt_sum"] / max(st["cnt"], 1) if st["cnt"] > 0 else float(row["TransactionAmt"])
    oh = [0.0] * 5
    oh[prod_map.get(row["ProductCD"], 0)] = 1.0
    amt = float(row["TransactionAmt"])
    feats1.append([
        np.log1p(amt), amt - int(amt), *oh,
        np.log1p(st["cnt"]), np.log1p(avg_a), eb_f,
        np.log1p(max(float(row["C1"]), 0.0)), np.log1p(max(float(row["D1"]), 0.0))
    ])
    st["cnt"] += 1
    st["pos"] += float(row["isFraud"])
    st["amt_sum"] += amt
    card_hist[c] = st

X1 = np.array(feats1, dtype=np.float32)
y1 = df1["isFraud"].values
s1 = df1["is_digital_nonlocal"].values
mu1, sd1 = X1[:i1_tr].mean(0), X1[:i1_tr].std(0) + 1e-6
X1_n = ((X1 - mu1) / sd1).astype(np.float32)

class VeriRailGate1(nn.Module):
    def __init__(self, tab_dim=12, hid=96):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(tab_dim, hid), nn.LayerNorm(hid), nn.GELU(),
            nn.Linear(hid, hid), nn.LayerNorm(hid), nn.GELU()
        )
        self.head = nn.Sequential(nn.Linear(hid, 48), nn.GELU(), nn.Dropout(0.2), nn.Linear(48, 1))
        self.adv = nn.Sequential(
            nn.utils.parametrizations.spectral_norm(nn.Linear(hid, 48)),
            nn.LeakyReLU(0.2),
            nn.utils.parametrizations.spectral_norm(nn.Linear(48, 1))
        )

    def forward(self, x, grl_lam=1.0):
        h = self.enc(x)
        return self.head(h).squeeze(-1), self.adv(GRLFn.apply(h, grl_lam)).squeeze(-1), h

m1 = VeriRailGate1(X1_n.shape[1]).to(device)
opt1 = torch.optim.AdamW(m1.parameters(), lr=1e-3, weight_decay=1e-4)
dl1 = DataLoader(
    TensorDataset(torch.tensor(X1_n[:i1_tr]), torch.tensor(y1[:i1_tr], dtype=torch.float32), torch.tensor(s1[:i1_tr], dtype=torch.float32)),
    batch_size=512, shuffle=True
)
pw1 = min((y1[:i1_tr] == 0).sum() / max((y1[:i1_tr] == 1).sum(), 1), 12.0)

for _ in range(12):
    m1.train()
    for bx, by, bs in dl1:
        bx, by, bs = bx.to(device), by.to(device), bs.to(device)
        lg, al, h = m1(bx, grl_lam=1.0)
        bce = F.binary_cross_entropy_with_logits(lg, by, reduction="none")
        loss = (torch.where(by == 1, pw1, 1.0) * ((1 - torch.exp(-bce)) ** 1.5) * bce).mean() + 0.5 * F.binary_cross_entropy_with_logits(al, bs) + 0.35 * dcor_loss(h, bs)
        opt1.zero_grad()
        loss.backward()
        opt1.step()

m1.eval()
with torch.no_grad():
    p1_cal = torch.sigmoid(m1(torch.tensor(X1_n[i1_tr:i1_cal]).to(device))[0]).cpu().numpy()
    p1_te = torch.sigmoid(m1(torch.tensor(X1_n[i1_cal:]).to(device))[0]).cpu().numpy()

y1_te, s1_te = y1[i1_cal:], s1[i1_cal:]
blunt_preds = ((s1_te == 1) | (df1["TransactionAmt"].values[i1_cal:] > 500)).astype(int)
blunt_fp_nonlocal = float(((blunt_preds == 1) & (y1_te == 0) & (s1_te == 1)).sum() / max(((y1_te == 0) & (s1_te == 1)).sum(), 1))

q1_by_alpha = {str(a): fit_mondrian(p1_cal, y1[i1_tr:i1_cal], s1[i1_tr:i1_cal], alpha=a) for a in [0.10, 0.07, 0.04]}
q1_10 = q1_by_alpha["0.1"]
vr_block = np.array([1 if ((1 - p1_te[i]) <= q1_10[str(s1_te[i])] and p1_te[i] > q1_10[str(s1_te[i])]) else 0 for i in range(len(p1_te))])
vr_fp_nonlocal = float(((vr_block == 1) & (y1_te == 0) & (s1_te == 1)).sum() / max(((y1_te == 0) & (s1_te == 1)).sum(), 1))

xgb1 = xgb.XGBClassifier(n_estimators=200, max_depth=5, scale_pos_weight=pw1, tree_method="hist", random_state=42).fit(X1_n[:i1_tr], y1[:i1_tr])
p1_xgb = xgb1.predict_proba(X1_n[i1_cal:])[:, 1]
xgb_fp_nonlocal = float(((p1_xgb > 0.5) & (y1_te == 0) & (s1_te == 1)).sum() / max(((y1_te == 0) & (s1_te == 1)).sum(), 1))

gate1_benchmarks = [
    {"model": "B1: Blunt Non-Local / >$500 Rule", "pr_auc": round(float(average_precision_score(y1_te, blunt_preds)), 4), "fp_freeze_nonlocal": round(blunt_fp_nonlocal, 4)},
    {"model": "B2: Temporal XGBoost", "pr_auc": round(float(average_precision_score(y1_te, p1_xgb)), 4), "fp_freeze_nonlocal": round(xgb_fp_nonlocal, 4)},
    {"model": "Full VeriRail Gate 1 (GRL+dCor+Mondrian)", "pr_auc": round(float(average_precision_score(y1_te, p1_te)), 4), "fp_freeze_nonlocal": round(vr_fp_nonlocal, 4)},
]

torch.save(m1.state_dict(), "artifacts/gate1_verirail.pt")
with open("artifacts/gate1_artifacts.json", "w") as f:
    json.dump({"mu": mu1.tolist(), "sd": sd1.tolist(), "q_thresholds_by_alpha": q1_by_alpha, "prior_fraud": g1_prior}, f, indent=2)

with open("artifacts/audit_results.json", "w") as f:
    json.dump({
        "gate2_benchmarks": gate2_benchmarks,
        "gate1_benchmarks": gate1_benchmarks,
        "fairness_audit": {
            "matched_pairs": len(rgaps),
            "real_gap": round(float(np.mean(rgaps)), 4) if rgaps else 0.0,
            "unmitigated_gap": round(float(np.mean(agaps)), 4) if agaps else 0.0,
            "verirail_gap": round(float(np.mean(fgaps)), 4) if fgaps else 0.0,
            "cov_g0": round(cov0, 3),
            "cov_g1": round(cov1, 3),
            "gate1_nonlocal_fp_blunt": round(blunt_fp_nonlocal, 4),
            "gate1_nonlocal_fp_verirail": round(vr_fp_nonlocal, 4),
        },
        "failures": failures,
    }, f, indent=2)

print(json.dumps(json.load(open("artifacts/audit_results.json")), indent=2))

100% 58.3M/58.3M [00:03<00:00, 17.2MB/s]

-rw-r--r-- 1 root root 652M Dec 11  2019 /tmp/train_transaction.csv
{
  "gate2_benchmarks": [
    {
      "model": "B1: Majority Rule (All 0)",
      "pr_auc": 0.1004,
      "precision": 0.0,
      "recall": 0.0,
      "f1": 0.0,
      "brier": 0.1004,
      "acc": 0.9
    },
    {
      "model": "B2: Company Trailing Rate",
      "pr_auc": 0.2895,
      "precision": 0.277,
      "recall": 0.588,
      "f1": 0.376,
      "brier": 0.0781,
      "acc": 0.804
    },
    {
      "model": "B3: TF-IDF + Tab XGBoost",
      "pr_auc": 0.3198,
      "precision": 0.301,
      "recall": 0.469,
      "f1": 0.367,
      "brier": 0.1014,
      "acc": 0.837
    },
    {
      "model": "Ablation: FiLM Only (No GRL/dCor)",
      "pr_auc": 0.3476,
      "precision": 0.328,
      "recall": 0.5,
      "f1": 0.396,
      "brier": 0.0936,
      "acc": 0.847
    },
    {
      "model": "Full VeriRail Gate 2 (FiLM+GRL+dCor)",
      "pr_auc": 0.3504,
      "precision":

In [12]:
# === COLAB CELL 6 (Zero-Leakage Integrity Checks & Zip Download) ===
assert df["date_received"].is_monotonic_increasing, "Gate 2 dates must be strictly increasing!"
assert df1["TransactionDT"].is_monotonic_increasing, "Gate 1 TransactionDT must be strictly increasing!"
for col in ["company_public_response", "timely_response", "consumer_disputed"]:
    assert col not in df.columns, f"Leaked column {col} found!"
assert not np.isnan(p_te2).any() and not np.isnan(p1_te).any(), "NaN probabilities detected!"
chk = pii_parity_scrub("On 03/14/2026 card 4111-2222-3333-4444 lost $1,250.00")
assert "XX/XX/XXXX" in chk and "{\\$1,250.00}" in chk and "XXXX-XXXX-XXXX-XXXX" in chk
print("ALL ZERO-LEAKAGE & INTEGRITY TESTS PASSED!")

!zip -r verirail_artifacts.zip artifacts/
from google.colab import files
files.download("verirail_artifacts.zip")

ALL ZERO-LEAKAGE & INTEGRITY TESTS PASSED!
  adding: artifacts/ (stored 0%)
  adding: artifacts/gate2_artifacts.json (deflated 75%)
  adding: artifacts/gate2_verirail.pt (deflated 9%)
  adding: artifacts/gate1_artifacts.json (deflated 53%)
  adding: artifacts/audit_results.json (deflated 65%)
  adding: artifacts/gate1_verirail.pt (deflated 12%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>